In [2]:
!pip install transformers datasets scikit-learn -q

In [3]:
from google.colab import files
uploaded = files.upload()

Saving test.json to test.json
Saving train.json to train.json


In [4]:
import json

with open("train.json") as f:
    train_data = json.load(f)

with open("test.json") as f:
    test_data = json.load(f)


print(type(train_data))
print(len(train_data))
print(train_data[0])

<class 'list'>
1864
{'id': 'train_00001', 'dialogue_id': 'ESConv_001', 'dialogue': [{'text': 'hi', 'speaker': 'seeker'}], 'current_text': 'hi', 'label': 0}


In [5]:
from collections import Counter

labels = [d['label'] for d in train_data]
print(Counter(sorted(labels)))

Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})


In [6]:
def format_input(example, k=5):

    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history

    context_text = " [SEP] ".join(
        [f"{t['speaker']}: {t['text']}" for t in last_k]
    )
    target = example['current_text']
    return f"{context_text} [SEP] TARGET: {target}"


print(format_input(train_data[1]))

seeker: hi [SEP] supporter: Hi. How's it goin'? Are you feeling troubled about something this morning? I know it can be hard to talk about personal problems sometimes. I have a lot of trouble with that myself. [SEP] TARGET: going smooth, i need to get a nice and a surprise able present for my parent, and at the same time I need to pay my house bills my father understands the situation that my earning is not that much, but my mum will fuck complain that I am taking care of my wife but not her


In [19]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch
from sklearn.model_selection import train_test_split
from huggingface_hub import notebook_login

notebook_login()

tokenizer = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased")

train_split, val_split = train_test_split(
    train_data, test_size=0.1, random_state=42,
    stratify=[d['label'] for d in train_data]
)

print(f"Train: {len(train_split)}, Val: {len(val_split)}")

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = format_input(self.data[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        result = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }
        if not self.is_test:
            result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

# rebuild datasets
train_dataset = DefenseDataset(train_split, tokenizer)
val_dataset   = DefenseDataset(val_split, tokenizer)
test_dataset  = DefenseDataset(test_data, tokenizer, is_test=True)  # is_test=True

Train: 1677, Val: 187


In [10]:
import numpy as np
from transformers import AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    "mental/mental-bert-base-uncased",
    num_labels=9
)

# class imbalance handle
label_list = [d['label'] for d in train_split]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(9),
    y=label_list
)
weights = torch.tensor(class_weights, dtype=torch.float).to('cuda')
print("Class weights:", weights)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Class weights: tensor([0.7005, 1.9210, 3.3879, 2.0936, 2.4518, 4.3333, 1.2022, 0.2139, 7.4533],
       device='cuda:0')


In [16]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'accuracy':    (preds == labels).mean()
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./mentalbert-psydef',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',       # fixed
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=50,
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted,Accuracy
1,4.166042,2.038542,0.146527,0.338950,0.358289
2,3.669078,1.899173,0.213975,0.483833,0.486631
3,3.270104,1.934266,0.182488,0.449560,0.470588
4,2.865314,1.794403,0.229531,0.506799,0.491979
5,2.569042,1.777492,0.222889,0.515770,0.502674


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=525, training_loss=3.3583936128162204, metrics={'train_runtime': 330.799, 'train_samples_per_second': 25.348, 'train_steps_per_second': 1.587, 'total_flos': 2206324858383360.0, 'train_loss': 3.3583936128162204, 'epoch': 5.0})

In [20]:
import zipfile

# predict
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)


for i, entry in enumerate(test_data):
    entry['label'] = int(preds[i])

# save
with open('prediction.json', 'w') as f:
    json.dump(test_data, f)

# zip
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('prediction.json')

print("Done. submission.zip ready.")
files.download('submission.zip')

Done. submission.zip ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>